# Project 1 – Titanic Dataset EDA & Visualization

**Student Name:** Manish Pawar

This notebook performs data loading, cleaning, EDA, visualizations, dashboard creation, and summary writing.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: use seaborn if available
try:
    import seaborn as sns
    sns.set_theme()
except Exception:
    sns = None

print("Student Name: Manish Pawar")
print("Libraries imported successfully")

## 1. Load Dataset

Upload `titanic.csv` in the same folder. If the file is not available, this notebook creates a sample Titanic-style dataset for practice.

In [ ]:
import os

if os.path.exists("titanic.csv"):
    df = pd.read_csv("titanic.csv")
else:
    np.random.seed(7)
    n = 891
    sex = np.random.choice(['male','female'], size=n, p=[0.647,0.353])
    pclass = np.random.choice([1,2,3], size=n, p=[0.242,0.207,0.551])
    age = np.clip(np.random.normal(29.7, 14.5, size=n), 0.42, 80)
    fare_base = {1:80,2:25,3:13}
    fare = np.array([np.random.gamma(2, fare_base[c]/2) for c in pclass])
    embarked = np.random.choice(['S','C','Q'], size=n, p=[0.72,0.19,0.09])
    prob = 0.12 + (sex=='female')*0.48 + (pclass==1)*0.20 + (pclass==2)*0.08 + (age<12)*0.15 - (age>60)*0.08
    prob = np.clip(prob, 0.03, 0.92)
    survived = np.random.binomial(1, prob)
    df = pd.DataFrame({
        'PassengerId': range(1,n+1),
        'Survived': survived,
        'Pclass': pclass,
        'Name': [f'Passenger {i}' for i in range(1,n+1)],
        'Sex': sex,
        'Age': np.round(age,1),
        'SibSp': np.random.choice([0,1,2,3,4], n, p=[0.68,0.22,0.06,0.03,0.01]),
        'Parch': np.random.choice([0,1,2,3], n, p=[0.76,0.14,0.08,0.02]),
        'Ticket': [f'TKT{i:05d}' for i in range(1,n+1)],
        'Fare': np.round(fare,2),
        'Cabin': np.nan,
        'Embarked': embarked
    })
    df.loc[np.random.choice(n, 177, replace=False), 'Age'] = np.nan
    df.loc[np.random.choice(n, 2, replace=False), 'Embarked'] = np.nan

df.head()

In [ ]:
# Dataset information
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nStatistical Summary:")
df.describe()

## 2. Missing Values Before Cleaning

In [ ]:
missing_before = df.isnull().sum()
missing_before

## 3. Data Cleaning

In [ ]:
df_clean = df.copy()

# Fill missing Age values using median
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# Fill missing Embarked values using mode
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Drop Cabin column because it has too many missing values
if 'Cabin' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Cabin'])

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# Create new feature
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1

df_clean.to_csv("titanic_cleaned.csv", index=False)

print("Missing values after cleaning:")
print(df_clean.isnull().sum())
print("\nCleaned file saved as titanic_cleaned.csv")

## 4. Exploratory Data Analysis

In [ ]:
print("Overall Survival Rate:", round(df_clean['Survived'].mean() * 100, 2), "%")
print("\nSurvival Rate by Gender:")
print((df_clean.groupby('Sex')['Survived'].mean() * 100).round(2))
print("\nSurvival Rate by Passenger Class:")
print((df_clean.groupby('Pclass')['Survived'].mean() * 100).round(2))

## 5. Visualizations

In [ ]:
plt.figure(figsize=(7,5))
df_clean.groupby('Sex')['Survived'].mean().plot(kind='bar')
plt.title('Survival Rate by Gender')
plt.ylabel('Survival Rate')
plt.xlabel('Gender')
plt.tight_layout()
plt.savefig('chart1_survival_by_gender.png', dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(df_clean['Age'], bins=25)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('chart2_age_distribution.png', dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
df_clean['Pclass'].value_counts().sort_index().plot(kind='pie', autopct='%1.1f%%')
plt.title('Passenger Class Distribution')
plt.ylabel('')
plt.tight_layout()
plt.savefig('chart3_class_distribution.png', dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
df_clean.boxplot(column='Fare', by='Pclass')
plt.title('Fare Distribution by Passenger Class')
plt.suptitle('')
plt.xlabel('Passenger Class')
plt.ylabel('Fare')
plt.tight_layout()
plt.savefig('chart4_fare_boxplot.png', dpi=160)
plt.show()

In [ ]:
corr = df_clean[['Survived','Pclass','Age','SibSp','Parch','Fare','FamilySize']].corr()

plt.figure(figsize=(8,6))
plt.imshow(corr)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation Heatmap')
plt.colorbar()

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('chart5_correlation_heatmap.png', dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(df_clean['Age'], df_clean['Fare'], alpha=0.6)
plt.title('Age vs Fare')
plt.xlabel('Age')
plt.ylabel('Fare')
plt.tight_layout()
plt.savefig('chart6_age_vs_fare_scatter.png', dpi=160)
plt.show()

## 6. Dashboard Image

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(14,8))

df_clean.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0,0], title='Survival by Gender')
axes[0,1].hist(df_clean['Age'], bins=20)
axes[0,1].set_title('Age Distribution')
df_clean['Pclass'].value_counts().sort_index().plot(kind='bar', ax=axes[0,2], title='Class Count')
df_clean.boxplot(column='Fare', by='Pclass', ax=axes[1,0])
axes[1,0].set_title('Fare by Class')
axes[1,1].imshow(corr)
axes[1,1].set_title('Correlation')
axes[1,1].set_xticks(range(len(corr.columns)))
axes[1,1].set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=7)
axes[1,1].set_yticks(range(len(corr.columns)))
axes[1,1].set_yticklabels(corr.columns, fontsize=7)
axes[1,2].scatter(df_clean['Age'], df_clean['Fare'], alpha=0.5)
axes[1,2].set_title('Age vs Fare')

plt.suptitle('Titanic EDA Dashboard - Manish Pawar')
plt.tight_layout()
plt.savefig('dashboard_titanic.png', dpi=160)
plt.show()

## 7. 3 Interesting Findings

1. Female passengers had a much higher survival rate than male passengers.
2. First-class passengers had better survival chances than second- and third-class passengers.
3. Higher fare values were generally linked with higher survival chances.

## 8. Written Summary

Student Name: Manish Pawar

In this project, I performed Exploratory Data Analysis (EDA) on the Titanic dataset using Python, Pandas, Matplotlib, and Seaborn. The dataset contained information about passengers such as age, gender, passenger class, fare, and survival status. First, I loaded the dataset and inspected its structure using functions like head(), dtypes(), and describe(). I identified missing values in the Age, Cabin, and Embarked columns. To clean the data, I filled missing Age values with the median age, filled missing Embarked values with the most common category, and removed the Cabin column because it contained too many missing values. Duplicate records were also checked and removed.

After cleaning, I explored the dataset to discover patterns and relationships. One important finding was that the overall survival rate was around 38%. Another interesting observation was that female passengers had a much higher survival rate than male passengers. Women survived at a much higher rate, while men had a lower survival rate. I also found that first-class passengers had better survival chances compared to passengers in lower classes.

Using Matplotlib and Seaborn, I created bar charts, histograms, pie charts, boxplots, heatmaps, and scatter plots to visualize the data. The correlation heatmap showed that passenger class and fare had a noticeable relationship with survival. Overall, this project helped me understand the complete EDA workflow, including data cleaning, statistical analysis, feature creation, visualization, and deriving meaningful insights from real-world datasets.